# Decision Models: Why Talk Is Expensive

Companion notebook for the 2026-10-10 post *"35x Faster, Zero Tokens: The Week Decision Models Went Mainstream"*.

We model four things:
1. The latency math behind OpenAI's "10x faster than the Responses API" claim.
2. The cost math for 1M classifications across TypeSafe Jev, the Decisions API, and plain Luna.
3. Calibration: why a decision model's probability is a product, not a vibe (Brier score).
4. The chaining trap: dependent decisions are separate requests, so latency adds up.


In [ ]:
# 1) Latency model: autoregressive generation vs single-pass decision scoring
# T_auto = prefill + n_out / token_rate ; T_decide ~= one full pass, no tokens out.

def t_autoregressive(n_out, prefill_ms=300, tps=40):
    return prefill_ms + 1000 * n_out / tps

def t_decide_ms(prefill_ms=300, head_ms=15):
    return prefill_ms + head_ms

t_auto = t_autoregressive(50)          # ~50 output tokens for a JSON answer
t_decide = t_decide_ms()
print(f"Responses-style classification: ~{t_auto:.0f} ms")
print(f"Single-pass decision scoring:   ~{t_decide:.0f} ms")
print(f"Speedup band: {t_auto / (t_decide + 0):.1f}x")
# Sanity: matches the observed bands (independent median 230 ms vs baseline 1,460 ms)
assert 1000 < t_auto < 2000, t_auto
assert t_decide < 500, t_decide
assert t_auto / t_decide > 3, "decision path should dominate generation by a wide margin"
print("OK: latency model reproduces the claimed bands")


In [ ]:
# 2) Cost math: 1M classifications, 500 input tokens each
# Prices (Oct 2026): Jev $0.042/1M in; Decisions API $0.10/1M in, $0 out; Luna Responses $0.10 in / $0.50 out.
N, TOK_IN, TOK_OUT = 1_000_000, 500, 20

def bill(n, tok_in, tok_out, p_in, p_out):
    return n * tok_in / 1e6 * p_in + n * tok_out / 1e6 * p_out

jev = bill(N, TOK_IN, 0, 0.042, 0)
decisions = bill(N, TOK_IN, 0, 0.10, 0)
responses = bill(N, TOK_IN, TOK_OUT, 0.10, 0.50)
print(f"TypeSafe Jev 1.13:        ${jev:,.0f}")
print(f"OpenAI Decisions API:     ${decisions:,.0f}")
print(f"GPT-6 Luna via Responses: ${responses:,.0f}")
assert abs(jev - 21.0) < 1e-9
assert abs(decisions - 50.0) < 1e-9
assert abs(responses - 60.0) < 1e-9
print("OK: cost bands match the post ($21 / $50 / $60)")


In [ ]:
# 3) Calibration: the confidence number is the product.
# Brier score = mean((p - outcome)^2). A model that says 0.9 and is right 60% of the time
# is worse than a calibrated one — and thresholds only work on calibrated numbers.
import numpy as np

rng = np.random.default_rng(0)
true_rate = 0.70
outcomes = (rng.random(100_000) < true_rate).astype(float)

p_calibrated = np.full_like(outcomes, true_rate)                 # honest 70%
p_overconfident = np.full_like(outcomes, 0.98)                  # LLM-style: "I'm 98% sure" — every time

def brier(p, y):
    return float(np.mean((p - y) ** 2))

b_cal, b_over = brier(p_calibrated, outcomes), brier(p_overconfident, outcomes)
print(f"Brier (calibrated 0.70): {b_cal:.4f}   (theoretical floor: {true_rate*(1-true_rate):.4f})")
print(f"Brier (overconfident):   {b_over:.4f}")
assert b_cal < b_over, "calibration must beat confident-wrongness"
assert abs(b_cal - true_rate * (1 - true_rate)) < 0.01
print("OK: calibrated probabilities score better — this is what RLCD optimizes")


In [ ]:
# 4) The chaining trap: dependent decisions cannot share one request.
# OpenAI: 'a decision that depends on an earlier answer needs a second request.' Latency adds.
MEDIAN_MS = 230  # independent Decisions API median

def sequential_latency(k, per_call_ms=MEDIAN_MS):
    return k * per_call_ms

for k in (1, 3, 5):
    print(f"{k} chained decisions -> {sequential_latency(k)} ms sequential "
          f"(vs ~{MEDIAN_MS} ms if independent and batched)")
assert sequential_latency(5) == 1150
print("OK: chaining is linear — design decision graphs to minimize depth")


## Takeaway

The latency win comes from deleting the token loop, not from a faster model. The cost win is mechanical (no output tokens to bill). But two things stay on you: **calibration** (thresholds need labeled history — build that set first) and **chaining** (dependent decisions pay full latency per step).
